# CIC-IDS2017: Packet-Level Feature Extraction & Label Mapping Workflow

This notebook performs the complete end-to-end pipeline:
1. **Understands & Configures** the `extractor.py` script to generate enriched packet-level and flow features from raw PCAPs.
2. **Executes Extraction** on `Monday-WorkingHours.pcap` to produce a Unified Dataset with extra packet-level features.
3. **Normalizes & Resolves** CICFlowMeter quirks (column whitespace, bidirectional endpoint flipping, timestamp formatting).
4. **Maps & Transfers Labels** from the original `Monday-WorkingHours.pcap_ISCX.csv` to the newly extracted Unified Dataset.
5. **Generates Full Matching Statistics** showing exact row counts, match percentage, and label verification.

### Step 1: Environment Setup & Path Configuration
Ensure `tshark` (Wireshark command-line engine) is found in your environment, and configure input/output paths.

In [11]:
import os
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd

# Configure paths (Update these if running on Colab or a different directory)
WORKSPACE_DIR = os.getcwd()
PCAP_FILE = os.path.join(WORKSPACE_DIR, "Monday-WorkingHours.pcap")
ORIGINAL_CSV = os.path.join(WORKSPACE_DIR, "Monday-WorkingHours.pcap_ISCX.csv")
UNIFIED_OUTPUT_CSV = os.path.join(WORKSPACE_DIR, "unified_Monday-WorkingHours.csv")
FINAL_LABELED_CSV = os.path.join(WORKSPACE_DIR, "unified_Monday-WorkingHours_labeled.csv")

# Check for tshark
# If on Windows and installed with Wireshark, add default path if not already in PATH:
wireshark_win_path = r"C:\Program Files\Wireshark"
if os.path.exists(wireshark_win_path) and wireshark_win_path not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + wireshark_win_path

tshark_bin = shutil.which("tshark")
if tshark_bin:
    print(f"[OK] tshark found at: {tshark_bin}")
else:
    print("[WARNING] tshark not found in PATH!")
    print("  - On Windows: Install Wireshark (https://www.wireshark.org/) and ensure tshark is checked.")
    print("  - On Colab / Ubuntu: Run `!apt-get update && apt-get install -y tshark`")

print(f"PCAP File exists: {os.path.exists(PCAP_FILE)} ({os.path.getsize(PCAP_FILE) / (1024**3):.2f} GB)")
print(f"Original ISCX CSV exists: {os.path.exists(ORIGINAL_CSV)} ({os.path.getsize(ORIGINAL_CSV) / (1024**2):.2f} MB)")

[OK] tshark found at: C:\Program Files\Wireshark\tshark.EXE
PCAP File exists: True (10.08 GB)
Original ISCX CSV exists: True (256.20 MB)


### Step 2: Extract Unified Flow + Packet-Level Features from PCAP
We invoke the core extraction logic from `extractor.py` configured for `PROFILE = "CICIDS2017"`.
This outputs the 80+ standard CICFlowMeter features PLUS packet-level statistics (TTL distribution, TCP window dynamics, higher-order payload moments: skewness/kurtosis, retransmission count, port-scan entropy).

In [12]:
import extractor

# Ensure temporary working directories exist locally
TMP_DIR = os.path.join(WORKSPACE_DIR, "tshark_tmp")
os.makedirs(TMP_DIR, exist_ok=True)
extractor.LOCAL_TMP_DIR = TMP_DIR
extractor.set_profile("CICIDS2017")

unified_parquet_path = os.path.join(TMP_DIR, "unified_Monday.parquet")

print(f"Starting feature extraction from {PCAP_FILE}...")
print("This processes packets in chunks via tshark and computes streaming statistical moments.")
# Note: Monday PCAP is ~10GB; processing time depends on CPU and disk speed.
# If already processed, you can skip re-extraction:
if not os.path.exists(UNIFIED_OUTPUT_CSV) and not os.path.exists(unified_parquet_path):
    total_flows = extractor.process_day(PCAP_FILE, unified_parquet_path)
    print(f"[Done] Extracted {total_flows:,} flows.")
    
    # Convert Parquet to CSV for seamless tabular handling
    df_ext = pd.read_parquet(unified_parquet_path)
    df_ext.to_csv(UNIFIED_OUTPUT_CSV, index=False)
    print(f"Saved unified CSV to: {UNIFIED_OUTPUT_CSV}")
else:
    print("[Info] Unified extracted file already exists, skipping raw PCAP parsing.")

Starting feature extraction from c:\Users\chall\OneDrive\Desktop\SIH\Monday-WorkingHours.pcap...
This processes packets in chunks via tshark and computes streaming statistical moments.
[Info] Unified extracted file already exists, skipping raw PCAP parsing.


### Step 3: Load Datasets & Clean Dataset Quirks
The released CIC-IDS2017 CSV files have several well-known quirks:
1. **Whitespace in column names**: E.g., `" Label"` and `" Destination Port"` have leading spaces. We strip all column names.
2. **Bidirectional Flow Direction**: Depending on which packet was captured first, `(Source IP, Destination IP)` can be flipped relative to the original Flow ID. We generate a direction-independent **Canonical 5-Tuple Key**.
3. **Timestamp Formats**: Timestamp strings are converted to normalized Unix epoch seconds.

In [13]:
print("Loading Original Labeled CSV...")
# Load original CSV (read stripped column names)
df_orig = pd.read_csv(ORIGINAL_CSV, low_memory=False)
df_orig.columns = df_orig.columns.str.strip()
print(f"Original CSV rows: {len(df_orig):,}, columns: {df_orig.shape[1]}")
print(f"Original Label distribution:\n{df_orig['Label'].value_counts()}\n")

print("Loading Unified Extracted CSV (or Parquet)...")
if os.path.exists(UNIFIED_OUTPUT_CSV):
    df_unified = pd.read_csv(UNIFIED_OUTPUT_CSV, low_memory=False)
elif os.path.exists(unified_parquet_path):
    df_unified = pd.read_parquet(unified_parquet_path)
else:
    raise FileNotFoundError("Unified extracted dataset not found!")

df_unified.columns = df_unified.columns.str.strip()
print(f"Unified CSV rows: {len(df_unified):,}, columns: {df_unified.shape[1]}")

Loading Original Labeled CSV...
Original CSV rows: 529,918, columns: 85
Original Label distribution:
BENIGN    529918
Name: Label, dtype: int64

Loading Unified Extracted CSV (or Parquet)...
Unified CSV rows: 529,683, columns: 124


### Step 4: Build Canonical Flow Keys & Timestamp Normalization
We create a canonical 5-tuple representation: `sort((IP1, Port1), (IP2, Port2)) + Protocol`.
This guarantees that forward and backward flows produce identical join keys regardless of who initiated the connection.

In [14]:
def build_canonical_key(df):
    """Vectorized generation of canonical 5-tuple key: min(ep1, ep2) <-> max(ep1, ep2) : proto"""
    src_ep = df["Source IP"].astype(str) + ":" + df["Source Port"].astype(str)
    dst_ep = df["Destination IP"].astype(str) + ":" + df["Destination Port"].astype(str)
    proto = df["Protocol"].astype(str)
    
    # Determine order
    is_src_smaller = src_ep <= dst_ep
    ep_low = np.where(is_src_smaller, src_ep, dst_ep)
    ep_high = np.where(is_src_smaller, dst_ep, src_ep)
    
    return ep_low + "<->" + ep_high + ":" + proto

# 1. Assign canonical keys
print("Generating canonical 5-tuple keys...")
df_orig["canon_key"] = build_canonical_key(df_orig)
df_unified["canon_key"] = build_canonical_key(df_unified)

# 2. Clean timestamp strings and build aligned epoch seconds
print("Normalizing timestamps to aligned epoch seconds...")
df_orig["ts_str"] = df_orig["Timestamp"].str.strip()
df_unified["ts_str"] = df_unified["Timestamp"].str.strip()

df_orig["t_sec"] = pd.to_datetime(df_orig["ts_str"], format="%d/%m/%Y %I:%M:%S").astype("int64") // 10**9
df_unified["t_sec"] = pd.to_datetime(df_unified["ts_str"], format="%d/%m/%Y %I:%M:%S").astype("int64") // 10**9

print("Keys and timestamps generated successfully.")

Generating canonical 5-tuple keys...
Normalizing timestamps to aligned epoch seconds...
Keys and timestamps generated successfully.


### Step 5: High-Precision Mapping & Label Transfer
We execute a two-stage matching strategy:
- **Stage 1 (Exact Match)**: Matches flows with identical `canon_key`, exact `epoch_sec`, and matching packet counts (`Total Fwd Packets`).
- **Stage 2 (Time-Window Match)**: For flows that differ by ±1–2 seconds due to microsecond rounding or clock offset, matches to the nearest flow within the same `canon_key`.

In [15]:
print("Executing High-Precision Flow Mapping...")

df_orig["orig_row_id"] = np.arange(len(df_orig))
df_unified["unified_row_id"] = np.arange(len(df_unified))

# Prepare lookup subset from original labeled CSV
orig_lookup = df_orig[["orig_row_id", "Flow ID", "Source IP", "Destination IP", "canon_key", "ts_str", "t_sec", "Total Fwd Packets", "Label"]].copy()

# Stage 1: Exact Match on (Flow ID, Source IP, Destination IP, ts_str, Total Fwd Packets)
m_exact = pd.merge(
    df_unified[["unified_row_id", "Flow ID", "Source IP", "Destination IP", "ts_str", "Total Fwd Packets"]],
    orig_lookup[["orig_row_id", "Flow ID", "Source IP", "Destination IP", "ts_str", "Total Fwd Packets", "Label"]],
    on=["Flow ID", "Source IP", "Destination IP", "ts_str", "Total Fwd Packets"],
    how="inner"
).drop_duplicates(subset=["unified_row_id"]).drop_duplicates(subset=["orig_row_id"])
print(f"Stage 1 (Exact match on Flow ID + IP + time + fwd pkts): {len(m_exact):,} rows matched ({len(m_exact)/len(df_orig)*100:.2f}%).")

# Stage 2: Match remaining on (canon_key, ts_str)
matched_unified_ids = set(m_exact["unified_row_id"])
matched_orig_ids = set(m_exact["orig_row_id"])

unmatched_unified = df_unified[~df_unified["unified_row_id"].isin(matched_unified_ids)][["unified_row_id", "canon_key", "ts_str"]]
unmatched_orig = orig_lookup[~orig_lookup["orig_row_id"].isin(matched_orig_ids)][["orig_row_id", "canon_key", "ts_str", "Label"]]

m_time = pd.merge(
    unmatched_unified,
    unmatched_orig,
    on=["canon_key", "ts_str"],
    how="inner"
).drop_duplicates(subset=["unified_row_id"]).drop_duplicates(subset=["orig_row_id"])
print(f"Stage 2 (Match on key + time): {len(m_time):,} additional rows matched.")

# Stage 3: Merge_asof with 2-second tolerance for remaining within same canon_key
matched_unified_ids.update(m_time["unified_row_id"])
matched_orig_ids.update(m_time["orig_row_id"])

rem_unified = df_unified[~df_unified["unified_row_id"].isin(matched_unified_ids)][["unified_row_id", "canon_key", "t_sec"]].sort_values("t_sec")
rem_orig = orig_lookup[~orig_lookup["orig_row_id"].isin(matched_orig_ids)][["orig_row_id", "canon_key", "t_sec", "Label"]].sort_values("t_sec")

if not rem_unified.empty and not rem_orig.empty:
    m_asof = pd.merge_asof(
        rem_unified,
        rem_orig,
        on="t_sec",
        by="canon_key",
        tolerance=2, # within 2 seconds
        direction="nearest"
    ).dropna(subset=["orig_row_id"]).drop_duplicates(subset=["unified_row_id"]).drop_duplicates(subset=["orig_row_id"])
    print(f"Stage 3 (Tolerance window match within 2s): {len(m_asof):,} additional rows matched.")
else:
    m_asof = pd.DataFrame(columns=["unified_row_id", "orig_row_id", "Label"])

# Combine all matched pairs
all_matches = pd.concat([
    m_exact[["unified_row_id", "orig_row_id", "Label"]],
    m_time[["unified_row_id", "orig_row_id", "Label"]],
    m_asof[["unified_row_id", "orig_row_id", "Label"]]
], ignore_index=True)

print(f"\n>>> TOTAL UNIQUE MATCHES FOUND: {len(all_matches):,} / {len(df_orig):,} ({len(all_matches)/len(df_orig)*100:.2f}%) <<<")

Executing High-Precision Flow Mapping...
Stage 1 (Exact match on Flow ID + IP + time + fwd pkts): 521,934 rows matched (98.49%).
Stage 2 (Match on key + time): 7,666 additional rows matched.
Stage 3 (Tolerance window match within 2s): 47 additional rows matched.

>>> TOTAL UNIQUE MATCHES FOUND: 529,647 / 529,918 (99.95%) <<<


### Step 6: Full Matching Statistics & Evaluation Report
Print exact numbers and percentages of rows matched, label preservation, and check coverage.

In [16]:
n_orig = len(df_orig)
n_unified = len(df_unified)
n_matched = len(all_matches)

pct_orig_matched = (n_matched / n_orig) * 100
pct_unified_matched = (n_matched / n_unified) * 100

print("=" * 65)
print("           CIC-IDS2017 FLOW MAPPING VALIDATION REPORT")
print("=" * 65)
print(f"Total Rows in Original Released CSV : {n_orig:>12,}")
print(f"Total Rows in Extracted Unified CSV : {n_unified:>12,}")
print(f"Total Successfully Matched Rows     : {n_matched:>12,}")
print(f"Percentage of Original CSV Matched  : {pct_orig_matched:>11.2f}%")
print(f"Percentage of Unified CSV Matched   : {pct_unified_matched:>11.2f}%")
print("-" * 65)
print("Label Distribution of Matched Flows:")
print(all_matches["Label"].value_counts())
print("=" * 65)

           CIC-IDS2017 FLOW MAPPING VALIDATION REPORT
Total Rows in Original Released CSV :      529,918
Total Rows in Extracted Unified CSV :      529,683
Total Successfully Matched Rows     :      529,647
Percentage of Original CSV Matched  :       99.95%
Percentage of Unified CSV Matched   :       99.99%
-----------------------------------------------------------------
Label Distribution of Matched Flows:
BENIGN    529647
Name: Label, dtype: int64


### Step 7: Assign Labels to Unified Dataset & Inspect Packet Features
Assign the matched `Label` back to `df_unified`, verify new packet-level features, and export the final labeled dataset.

In [17]:
# Assign Label to df_unified
label_map = dict(zip(all_matches["unified_row_id"], all_matches["Label"]))
df_unified["Label"] = df_unified["unified_row_id"].map(label_map)

# Fill any unmatched flows (if any exist, mark or default to BENIGN for Monday)
unlabeled_count = df_unified["Label"].isna().sum()
if unlabeled_count > 0:
    print(f"Note: {unlabeled_count:,} flows in unified did not find a direct counterpart in original ISCX CSV.")
    print("For Monday, all ground truth traffic is BENIGN. Imputing remaining with BENIGN.")
    df_unified["Label"] = df_unified["Label"].fillna("BENIGN")

# Cleanup helper columns before saving
cols_to_drop = [c for c in ["unified_row_id", "canon_key", "epoch_sec", "ts_str", "t_sec"] if c in df_unified.columns]
df_final = df_unified.drop(columns=cols_to_drop)

# Highlight newly added packet-level features
packet_level_cols = [
    "pkt_ttl_mean", "pkt_ttl_std", "pkt_win_mean", "pkt_win_std",
    "pkt_payload_skew", "pkt_payload_kurtosis", "pkt_payload_nonzero_ratio",
    "retransmission_count", "syn_packet_count", "port_scan_entropy"
]
present_packet_cols = [c for c in packet_level_cols if c in df_final.columns]

print(f"\nSample of final dataset with new packet-level features and mapped Label:")
display_cols = ["Flow ID", "Label"] + present_packet_cols[:5]
print(df_final[display_cols].head())

# Save final labeled unified CSV
print(f"\nSaving final labeled unified dataset to: {FINAL_LABELED_CSV} ...")
df_final.to_csv(FINAL_LABELED_CSV, index=False)
print(f"[Done] Successfully saved {len(df_final):,} labeled flows with packet-level features!")


Note: 36 flows in unified did not find a direct counterpart in original ISCX CSV.
For Monday, all ground truth traffic is BENIGN. Imputing remaining with BENIGN.

Sample of final dataset with new packet-level features and mapped Label:
                                  Flow ID   Label  pkt_ttl_mean  pkt_ttl_std  \
0   192.168.10.5-8.254.250.126-49188-80-6  BENIGN          55.0          0.0   
1   192.168.10.5-8.254.250.126-49188-80-6  BENIGN          55.0          0.0   
2   192.168.10.5-8.254.250.126-49188-80-6  BENIGN          55.0          0.0   
3   192.168.10.5-8.254.250.126-49188-80-6  BENIGN          55.0          0.0   
4  192.168.10.14-8.253.185.121-49486-80-6  BENIGN          55.0          0.0   

   pkt_win_mean  pkt_win_std  pkt_payload_skew  
0         329.0          0.0               0.0  
1         329.0          0.0               0.0  
2         329.0          0.0               0.0  
3         329.0          0.0               0.0  
4         245.0          0.0          